In [62]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [63]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [64]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [65]:
train_data.sample(6)

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
6422,13716070,Paul: Lucky you!\r\nJohn: ?\r\nPete: Our class...,"John, Pete and Paul's classes have been cancel..."


In [66]:
train_data.shape

(14732, 3)

In [67]:
val_data.shape

(818, 3)

In [68]:
# random sampling 
train_data=train_data.sample(4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(500,random_state=42).reset_index(drop=True)

In [69]:
train_data.shape
val_data.shape

(500, 3)

### Data Preprocessing 

In [70]:
import re

In [71]:
def clean_data(text):
    text=re.sub(r"\r\n"," ",text) # lines
    text=re.sub(r"\s+", " ",text) # Spaces
    text=re.sub(r"<.*?>"," ",text) # html tags 
    text.strip()
    text.lower()
    return text

In [72]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)



In [73]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

### Tokenize

In [74]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [75]:
# raw data => tokenized inputs for fine tuning 

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"]
    return inputs

In [76]:
train_dataset=train_data.apply(tokenize,axis=1).to_list()

In [77]:
val_dataset=val_data.apply(tokenize,axis=1).to_list()

In [78]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [79]:
# input ids

# 1 => EOS, 0 => padding

# attention masks => Indicates which are the valid ids 

# labels-targets => summary token

In [80]:
len(train_dataset[0]["input_ids"])

512

In [81]:
len(train_dataset[0]["labels"])

150

In [82]:
type(train_dataset)

list

In [83]:
type(val_dataset)

list

### Working with our Model

In [84]:
# NLP => Generation task 

model=T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Fine-Tune

In [85]:
import torch
from transformers import is_av_available

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

In [86]:
print(device)

cuda


In [87]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [88]:
# Training Arguments

from numpy import save


training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr in 500

)

In [89]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [90]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.074091,0.391735
2,0.407223,0.366080
3,0.383945,0.358481
4,0.371874,0.356084
5,0.364441,0.355007
6,0.361392,0.354678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9938277943929037, metrics={'train_runtime': 878.2688, 'train_samples_per_second': 27.326, 'train_steps_per_second': 3.416, 'total_flos': 3248203235328000.0, 'train_loss': 0.9938277943929037, 'epoch': 6.0})

In [93]:
# load model => fine tune => save tune the model

In [94]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [95]:
model.from_pretrained("./saved_summary_model")
tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='./saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False